# 04 — Generic Supersede-Pattern Demo

Companion notebook to `09-development-challenge-data-lifecycle-and-versioning-gaps.md`. Chapter 09's
step 4 ("propose a concrete mechanism") names this shape directly: *"a stable group/logical identity
for 'all versions of this same thing,' a `supersedes`/`superseded-by` pointer from each version to the
one before or after it, and a well-defined 'get current version' query... See
`notebooks/04_generic_supersede_pattern_demo.ipynb` in this folder for a small, fully generic,
runnable implementation of exactly that shape."*

This notebook builds that one small, **domain-agnostic** data structure and query — not a document
versioner, not a baseline versioner, not a claims-library versioner specifically, but the single
reusable shape chapter 09 argues underlies **every** project-specific version of this problem named in
its course-by-course table (a document in course 5, a monitoring baseline in course 4, a pinned
service version in course 6, a reprocessed page in course 7, a vector chunk in course 8, an approved
claim in course 9, a detection result in course 10, a protocol amendment in course 11). Section 5
instantiates the same generic function against several of those domains directly, to make that claim
concrete rather than asserted.

Fully offline: standard library and (optionally) numpy only, no project-specific dependencies.

In [1]:
from dataclasses import dataclass
from typing import Optional, Dict, List

print("Imports OK")

Imports OK


## Step 1 — The generic entity: `id`, `version`, `supersedes_id`

Deliberately minimal: three fields, plus a `payload` for whatever domain-specific content a real
version of this entity would actually carry (irrelevant to the versioning mechanism itself, which is
exactly the point -- the mechanism doesn't need to know or care what `payload` holds). `supersedes_id`
is `None` for a first-ever version of something, and points at the `id` of the version it directly
replaces otherwise -- one link in a chain, not a jump straight to "the original."

In [2]:
@dataclass
class VersionedEntity:
    id: str
    version: int
    supersedes_id: Optional[str]   # the id of the version this one directly replaces, or None
    payload: dict


class SupersedeStore:
    """A tiny in-memory store -- stands in for whatever real table/collection a production system
    would keep these records in. All the actual versioning logic lives in get_current(), not here."""
    def __init__(self):
        self._by_id: Dict[str, VersionedEntity] = {}

    def add(self, entity: VersionedEntity) -> VersionedEntity:
        if entity.supersedes_id is not None and entity.supersedes_id not in self._by_id:
            raise ValueError(f"{entity.id} claims to supersede {entity.supersedes_id!r}, which doesn't exist yet")
        self._by_id[entity.id] = entity
        return entity

    def get(self, entity_id: str) -> VersionedEntity:
        return self._by_id[entity_id]

    def all(self) -> List[VersionedEntity]:
        return list(self._by_id.values())


print("VersionedEntity, SupersedeStore defined")

VersionedEntity, SupersedeStore defined


## Step 2 — `get_current_version`: follow the chain to the latest, from ANY id in it

The core claim this notebook exists to demonstrate: querying an **old** version's id must resolve to
the **current** version, not just return that old record unchanged. The implementation builds a
reverse index (`supersedes_id -> id`, i.e. "what directly replaced this id") once, then walks forward
from whatever id was asked about until it reaches an id that nothing in the store points back to --
the one entity in the chain that "nothing else supersedes," which is exactly chapter 09's own
definition of "current."

In [3]:
def get_current_version(entity_id: str, store: SupersedeStore) -> VersionedEntity:
    """Follows supersedes_id pointers FORWARD (old -> new) from entity_id until reaching the
    version nothing in the store supersedes -- the current one. Works identically no matter which
    id in the chain you start from."""
    # superseded_by[x] = the id of the entity whose supersedes_id points at x
    superseded_by = {e.supersedes_id: e.id for e in store.all() if e.supersedes_id is not None}

    current_id = entity_id
    seen = {current_id}
    while current_id in superseded_by:
        current_id = superseded_by[current_id]
        if current_id in seen:
            raise ValueError(f"cycle detected in supersede chain at {current_id!r} -- this must never happen")
        seen.add(current_id)
    return store.get(current_id)


def describe_lookup(entity_id: str, store: SupersedeStore) -> str:
    queried = store.get(entity_id)
    current = get_current_version(entity_id, store)
    if queried.id == current.id:
        return f"{entity_id} (v{queried.version}) -> already current"
    return f"{entity_id} (v{queried.version}) -> current is {current.id} (v{current.version})"


print("get_current_version, describe_lookup defined")

get_current_version, describe_lookup defined


## Step 3 — A three-version chain, queried from every point in it

`A` (v1) is superseded by `B` (v2), which is superseded by `C` (v3). The claim to check: looking up
`A`, `B`, or `C` must all resolve to `C` -- an old version's id staying resolvable, and always
resolving forward, is the entire point of the pointer (versus, say, deleting old records outright, or
leaving a stale search index with no way to redirect a hit on `A` to what's actually current).

In [4]:
store = SupersedeStore()
store.add(VersionedEntity(id="A", version=1, supersedes_id=None, payload={"note": "first draft"}))
store.add(VersionedEntity(id="B", version=2, supersedes_id="A", payload={"note": "corrected a typo"}))
store.add(VersionedEntity(id="C", version=3, supersedes_id="B", payload={"note": "updated after review"}))

for entity_id in ["A", "B", "C"]:
    print(describe_lookup(entity_id, store))

assert get_current_version("A", store).id == "C"
assert get_current_version("B", store).id == "C"
assert get_current_version("C", store).id == "C"
print()
print("Confirmed: querying the OLDEST version's id ('A') still resolves to the CURRENT version ('C') "
      "-- exactly the behavior a search index, a dashboard, or a downstream consumer needs from a hit "
      "on a version that's since been superseded twice over.")

A (v1) -> current is C (v3)
B (v2) -> current is C (v3)
C (v3) -> already current

Confirmed: querying the OLDEST version's id ('A') still resolves to the CURRENT version ('C') -- exactly the behavior a search index, a dashboard, or a downstream consumer needs from a hit on a version that's since been superseded twice over.


## Step 4 — Two independent chains never cross-contaminate, and an unsuperseded entity resolves to itself

Two unrelated entities (`X` and a completely separate, never-superseded `Y`) coexist in the same
store. The pointer-based design means there's no ambiguity between them -- each chain is walked
purely by following `supersedes_id`/`superseded_by` links specific to it, with no shared "group id"
required for correctness (chapter 09 mentions a group identity as one *option*; a pure supersede-chain
walk, as implemented here, is sufficient on its own).

In [5]:
store.add(VersionedEntity(id="X", version=1, supersedes_id=None, payload={"note": "unrelated entity, superseded once"}))
store.add(VersionedEntity(id="X2", version=2, supersedes_id="X", payload={"note": "X's correction"}))
store.add(VersionedEntity(id="Y", version=1, supersedes_id=None, payload={"note": "never superseded at all"}))

print(describe_lookup("X", store))
print(describe_lookup("X2", store))
print(describe_lookup("Y", store))

assert get_current_version("X", store).id == "X2"
assert get_current_version("Y", store).id == "Y", "an entity with no successor must resolve to itself, not error"
assert get_current_version("A", store).id == "C", "the original A/B/C chain must be unaffected by adding X/X2/Y"
print()
print("Confirmed: chains stay independent, and an entity that's never been superseded correctly "
      "resolves to itself rather than raising or returning nothing.")

X (v1) -> current is X2 (v2)
X2 (v2) -> already current
Y (v1) -> already current

Confirmed: chains stay independent, and an entity that's never been superseded correctly resolves to itself rather than raising or returning nothing.


## Step 5 — The same function, applied to several project-specific domains at once

This is the point of the notebook: `get_current_version()` above never referenced documents,
baselines, claims, or detections -- it only ever touched `id` and `supersedes_id`. Below, the exact
same function runs against three of chapter 09's own table rows, each with domain-appropriate
`payload` content and nothing else different, to make "this is the reusable shape underlying every
project-specific version of this problem" a checkable claim instead of an assertion.

In [6]:
domain_store = SupersedeStore()

# Course 5 (Document Uploader Service): a re-uploaded, corrected document.
domain_store.add(VersionedEntity(id="doc-refund-policy-v1", version=1, supersedes_id=None,
                                   payload={"domain": "course 5: uploaded document",
                                            "title": "Refund Policy", "text": "Refunds within 14 days."}))
domain_store.add(VersionedEntity(id="doc-refund-policy-v2", version=2, supersedes_id="doc-refund-policy-v1",
                                   payload={"domain": "course 5: uploaded document",
                                            "title": "Refund Policy", "text": "Refunds within 30 days."}))

# Course 4 (Model Risk Monitoring): a baseline retired when a new assistant version stabilizes.
domain_store.add(VersionedEntity(id="baseline-hsbc-faithfulness-tagA", version=1, supersedes_id=None,
                                   payload={"domain": "course 4: monitoring baseline",
                                            "assistant_version_tag": "tagA", "status_at_creation": "ACTIVE"}))
domain_store.add(VersionedEntity(id="baseline-hsbc-faithfulness-tagB", version=2,
                                   supersedes_id="baseline-hsbc-faithfulness-tagA",
                                   payload={"domain": "course 4: monitoring baseline",
                                            "assistant_version_tag": "tagB", "status_at_creation": "LEARNING"}))

# Course 9 (Claim Extraction & Tagging): an approved-claims-library entry revised after a label update.
domain_store.add(VersionedEntity(id="claim-lib-entry-88-v1", version=1, supersedes_id=None,
                                   payload={"domain": "course 9: approved claims library",
                                            "claim_text": "reduces symptom onset by 40%"}))
domain_store.add(VersionedEntity(id="claim-lib-entry-88-v2", version=2, supersedes_id="claim-lib-entry-88-v1",
                                   payload={"domain": "course 9: approved claims library",
                                            "claim_text": "reduces symptom onset by 40% in adults"}))

for old_id, expected_domain in [
    ("doc-refund-policy-v1", "course 5: uploaded document"),
    ("baseline-hsbc-faithfulness-tagA", "course 4: monitoring baseline"),
    ("claim-lib-entry-88-v1", "course 9: approved claims library"),
]:
    current = get_current_version(old_id, domain_store)
    print(f"[{expected_domain}] query on OLDEST id -> current is {current.id} (v{current.version}), "
          f"payload={current.payload}")
    assert current.payload["domain"] == expected_domain
    assert current.id != old_id, "each of these old ids should resolve to a DIFFERENT, newer id"

print()
print("Confirmed: the identical get_current_version() function correctly resolves a stale document id, "
      "a retired monitoring baseline id, and a superseded claims-library entry id -- three different "
      "courses' domain-specific versioning problems, one reusable mechanism.")

[course 5: uploaded document] query on OLDEST id -> current is doc-refund-policy-v2 (v2), payload={'domain': 'course 5: uploaded document', 'title': 'Refund Policy', 'text': 'Refunds within 30 days.'}
[course 4: monitoring baseline] query on OLDEST id -> current is baseline-hsbc-faithfulness-tagB (v2), payload={'domain': 'course 4: monitoring baseline', 'assistant_version_tag': 'tagB', 'status_at_creation': 'LEARNING'}
[course 9: approved claims library] query on OLDEST id -> current is claim-lib-entry-88-v2 (v2), payload={'domain': 'course 9: approved claims library', 'claim_text': 'reduces symptom onset by 40% in adults'}

Confirmed: the identical get_current_version() function correctly resolves a stale document id, a retired monitoring baseline id, and a superseded claims-library entry id -- three different courses' domain-specific versioning problems, one reusable mechanism.


## Tying it back

- Steps 1-2 are the whole mechanism: three fields (`id`, `version`, `supersedes_id`) and one query
  function that walks the pointer chain forward to whatever nothing else supersedes.
- Step 3 checks the one property that actually matters in production: a stale id (a search-index hit,
  a cached reference, a link in an old report) must still resolve to the current version, not silently
  return outdated content or a dead end.
- Step 4 confirms the design needs no extra "group id" bookkeeping to stay correct across unrelated
  entities, and correctly treats "never superseded" as a normal, first-class case, not a special one.
- Step 5 is chapter 09's actual thesis, made runnable: the same eleven lines of `get_current_version()`
  correctly serve three structurally unrelated domains from three different courses, which is exactly
  the argument for treating this as one pattern worth recognizing on the spot, not nine separate
  memorized answers.